# Baseline: import, histograms, segmentation

1. Import the image (config `input`) and save it to `.zarr`.
2. Per-pixel histograms of every channel.
3. Every segmentation listed under `segmentation` in the config (e.g. DAPI only, membrane only,
   membrane + DAPI), each with mean, sum and max intensity of every channel per object.

Runs on Mewtwo (open the repo with `mewtwo-code`). Paths come from the config.
The same steps run without the notebook with `python -m macroseg.baseline <config>`;
view the result on the Mac with `macroseg-view <name>`.

In [ ]:
import shutil
from pathlib import Path

from spatialdata import read_zarr

from macroseg.baseline import measure, segment
from macroseg.config import load_config
from macroseg.histograms import channel_summary, plot_histograms, sample_pixels
from macroseg.ingest import ingest

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
cfg = load_config(ROOT / "configs" / "exp201_mouse_neuroblastoma.yaml")

OVERWRITE = False  # True rebuilds the .zarr from the raw image

zarr_path = Path(cfg.zarr)
element = cfg.image.element

## 1. Import

In [ ]:
if zarr_path.exists() and not OVERWRITE:
    print(f"Reusing {zarr_path}")
else:
    zarr_path.parent.mkdir(parents=True, exist_ok=True)
    if zarr_path.exists():
        shutil.rmtree(zarr_path)
    ingest(cfg.input, element).write(zarr_path)
sdata = read_zarr(zarr_path)
sdata

## 2. Pixel histograms
Log axes; dashed lines at the 1st, 50th and 99th percentiles. Large images are subsampled on a regular grid.

In [ ]:
pixels = sample_pixels(sdata, element)
plot_histograms(pixels);
channel_summary(pixels)

## 3. Segmentations
One Cellpose run per entry under `segmentation` in the config; each writes a labels element
(same name) and a table `<name>_intensities`.

In [ ]:
for step in cfg.segmentation:
    print(f"{step.labels}: {list(step.channels)}")
    sdata = segment(sdata, element, step)
    sdata = measure(sdata, element, step.labels)

{step.labels: sdata.tables[f"{step.labels}_intensities"].n_obs for step in cfg.segmentation}